In [ ]:
# Load raw dataset 
# Handle missing values 
# Remove duplicates 
# Fix data types 
# Normalize text 
# Feature selection 
# Save cleaned dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os
from collections import Counter

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

df_recipes = pd.read_csv('../data/raw/RAW_recipes.csv')     # Đọc file dữ liệu món ăn từ thư mục raw
df_interactions = pd.read_csv('../data/raw/RAW_interactions.csv')

print(f"Recipes: {df_recipes.shape[0]} dòng x {df_recipes.shape[1]} cột")
print(f"Interactions: {df_interactions.shape[0]} dòng x {df_interactions.shape[1]} cột")

Recipes: 231637 dòng x 12 cột
Interactions: 1132367 dòng x 5 cột


In [3]:
nutrition_cols = [
    "calories",
    "fat",
    "sugar",
    "sodium",
    "protein",
    "saturated_fat",
    "carbs"
]

df_recipes[nutrition_cols] = pd.DataFrame(
    df_recipes['nutrition'].apply(ast.literal_eval).tolist(),
    index=df_recipes.index
)

df_recipes[nutrition_cols].describe()

,calories,fat,sugar,sodium,protein,saturated_fat,carbs
count,231637.00,231637.00,231637.00,231637.00,231637.00,231637.00,231637.00
mean,473.94,36.08,84.30,30.15,34.68,45.59,15.56
std,1189.71,77.80,800.08,131.96,58.47,98.24,81.82
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,174.40,8.00,9.00,5.00,7.00,7.00,4.00
50%,313.40,20.00,25.00,14.00,18.00,23.00,9.00
75%,519.70,41.00,68.00,33.00,51.00,52.00,16.00
max,434360.20,17183.00,362729.00,29338.00,6552.00,10395.00,36098.00


In [4]:
# Handle missing values

df_recipes.isnull().sum()
(df_recipes.isnull().sum() / len(df_recipes)) * 100
# => Chỉ có 2% dữ liệu Null ở cột Description  ==> trước mắt để "No description available" các description NULL

df_recipes['description'] = df_recipes['description'].fillna("No description available")


In [9]:
# Remove duplicates
#Duplicate records can occur due to data collection or merging processes. These duplicates can negatively affect data analysis and model training.

#   Kiểm tra duplicates
df_recipes[df_recipes.duplicated()]
df_interactions[df_interactions.duplicated()]

df_recipes[df_recipes['id'].duplicated()]
df_recipes['name'].duplicated().sum()

df_recipes['name'] = (
    df_recipes['name']
    .str.lower()          
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df_recipes[df_recipes['name'].duplicated()].head(30)

#lưu lại số lượng ban đầu để trace
count_recipes_old = len(df_recipes)
count_interactions_old = len(df_interactions)
#   Phân tích duplicates
#   Xóa duplicates
df_interactions = df_interactions.drop_duplicates(
    subset=['user_id', 'recipe_id', 'date', 'rating', 'review'],
    keep='first'
)

#xóa duplicates recipes theo id
df_recipes = df_recipes.drop_duplicates(subset=['id'], keep='first')

#   Kiểm tra lại dataset sau khi xóa
#reset lại số lượng
count_recipes_new = len(df_recipes)
count_interactions_new = len(df_interactions)

removed_recipes = count_recipes_old - count_recipes_new
removed_interactions = count_interactions_old - count_interactions_new


print("===== DUPLICATE CLEANING SUMMARY =====")
print(f"Recipes:      {count_recipes_old:,} -> {count_recipes_new:,} (removed {removed_recipes:,})")
print(f"Interactions: {count_interactions_old:,} -> {count_interactions_new:,} (removed {removed_interactions:,})")

#recheck
dup_recipe_id = df_recipes['id'].duplicated().sum()
dup_interaction_key = df_interactions.duplicated(subset=['user_id','recipe_id','date']).sum()
dup_name_after_norm = df_recipes['name'].duplicated().sum()

print("\n===== RECHECK =====")
print(f"Duplicate recipe id còn lại: {dup_recipe_id}")
print(f"Duplicate interaction key (user_id, recipe_id, date) còn lại: {dup_interaction_key}")
print(f"Duplicate recipe name sau normalize (tham khảo, KHÔNG tự động drop): {dup_name_after_norm}")

print("\nTop 5 tên recipe trùng sau normalize (nếu có):")
display(df_recipes[df_recipes['name'].duplicated(keep=False)].head(5))


===== DUPLICATE CLEANING SUMMARY =====
Recipes:      231,637 -> 231,637 (removed 0)
Interactions: 1,132,367 -> 1,132,367 (removed 0)

===== RECHECK =====
Duplicate recipe id còn lại: 0
Duplicate interaction key (user_id, recipe_id, date) còn lại: 0
Duplicate recipe name sau normalize (tham khảo, KHÔNG tự động drop): 2267

Top 5 tên recipe trùng sau normalize (nếu có):


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,calories,fat,sugar,sodium,protein,saturated_fat,carbs
15,chicken lickin good pork chops,63986,500,14664,2003-06-06,"['weeknight', 'time-to-make', 'course', 'main-ingredient...","[105.7, 8.0, 0.0, 26.0, 5.0, 4.0, 3.0]",5,"['dredge pork chops in mixture of flour , salt , dry mus...",here's and old standby i enjoy from time to time. it's f...,"['lean pork chops', 'flour', 'salt', 'dry mustard', 'gar...",7,105.70,8.00,0.00,26.00,5.00,4.00,3.00
16,chile rellenos,43026,45,52268,2002-10-14,"['60-minutes-or-less', 'time-to-make', 'course', 'main-i...","[94.0, 10.0, 0.0, 11.0, 11.0, 21.0, 0.0]",9,"['drain green chiles', 'sprinkle cornstarch on sheet pan...",a favorite from a local restaurant no longer in business...,"['egg roll wrap', 'whole green chilies', 'cheese', 'corn...",5,94.00,10.00,0.00,11.00,11.00,21.00,0.00
18,chinese chop suey,8559,70,4481,2001-01-27,"['weeknight', 'time-to-make', 'course', 'main-ingredient...","[395.4, 31.0, 20.0, 29.0, 51.0, 33.0, 8.0]",8,"['brown ground meat and onion in a large pot', 'add 1 / ...",easy one-pot dinner.,"['celery', 'onion', 'ground pork', 'soy sauce', 'beef br...",7,395.40,31.00,20.00,29.00,51.00,33.00,8.00
19,cream of cauliflower soup vegan,23850,110,3288,2002-03-28,"['lactose', 'weeknight', 'time-to-make', 'course', 'main...","[174.2, 4.0, 24.0, 1.0, 15.0, 1.0, 10.0]",10,['heat the oil or margarine in a soup pot and add the on...,this is a dairy free,"['canola oil', 'onion', 'garlic', 'cauliflower', 'potato...",16,174.20,4.00,24.00,1.00,15.00,1.00,10.00
20,cream of spinach soup,76808,45,95743,2003-11-17,"['60-minutes-or-less', 'time-to-make', 'course', 'main-i...","[126.0, 11.0, 2.0, 14.0, 5.0, 23.0, 4.0]",9,"['bring water and salt to a boil', 'cut the potatoes in ...","wonderful comfort food from rozanne gold, a favorite coo...","['water', 'salt', 'boiling potatoes', 'fresh spinach lea...",8,126.00,11.00,2.00,14.00,5.00,23.00,4.00


In [11]:
#Fix data types + lọc dữ liệu
df_interactions['date'] = pd.to_datetime(df_interactions['date'])
df_interactions['rating'] = pd.to_numeric(df_interactions['rating'], errors='coerce')

#Dòng nào bị NaN thì xóa
df_interactions = df_interactions.dropna(subset=['user_id','recipe_id','date','rating'])

#lọc range hợp lệ
df_interactions = df_interactions[df_interactions['rating'].between(1, 5)]

if 'minutes' in df_recipes.columns:
    df_recipes = df_recipes[df_recipes['minutes'] > 0]
if 'n_ingredients' in df_recipes.columns:
    df_recipes = df_recipes[df_recipes['n_ingredients'] > 0]

    # reset index
df_interactions = df_interactions.reset_index(drop=True)
df_recipes = df_recipes.reset_index(drop=True)

# summary
print("===== AFTER TYPE FIX + INVALID FILTER =====")
print("Interactions shape:", df_interactions.shape)
print("Recipes shape:", df_recipes.shape)
print("Unique users:", df_interactions['user_id'].nunique())
print("Unique recipes in interactions:", df_interactions['recipe_id'].nunique())
print("\nRating distribution:")
print(df_interactions['rating'].value_counts().sort_index())


===== AFTER TYPE FIX + INVALID FILTER =====
Interactions shape: (1071520, 5)
Recipes shape: (230543, 19)
Unique users: 196098
Unique recipes in interactions: 226590

Rating distribution:
rating
1     12818
2     14123
3     40855
4    187360
5    816364
Name: count, dtype: int64


In [12]:
# ===== STEP: Join consistency + split + save =====
# Mục tiêu:
# 1) Chỉ giữ interactions có recipe tồn tại trong recipes
# 2) Tách train/validation/test theo thời gian
# 3) Lưu file để train model

# 1) Join consistency: giữ interactions có recipe hợp lệ
valid_recipe_ids = set(df_recipes['id'].unique())
before_join = len(df_interactions)

df_interactions = df_interactions[df_interactions['recipe_id'].isin(valid_recipe_ids)].copy()
df_interactions = df_interactions.sort_values('date').reset_index(drop=True)

after_join = len(df_interactions)
print("===== JOIN CONSISTENCY =====")
print(f"Interactions before join filter: {before_join:,}")
print(f"Interactions after join filter : {after_join:,}")
print(f"Removed (recipe_id not in df_recipes): {before_join - after_join:,}")

# 2) Time-based split (80/10/10)
n = len(df_interactions)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

interactions_train = df_interactions.iloc[:train_end].copy()
interactions_val = df_interactions.iloc[train_end:val_end].copy()
interactions_test = df_interactions.iloc[val_end:].copy()

print("\n===== SPLIT SUMMARY =====")
print(f"Train: {len(interactions_train):,}")
print(f"Val  : {len(interactions_val):,}")
print(f"Test : {len(interactions_test):,}")

# 3) Tạo mapping user/item index (cho train recommender)
all_user_ids = pd.Index(df_interactions['user_id'].unique())
all_recipe_ids = pd.Index(df_interactions['recipe_id'].unique())

user2idx = {u: i for i, u in enumerate(all_user_ids)}
item2idx = {r: i for i, r in enumerate(all_recipe_ids)}

# gán u/i vào từng split
for _df in [interactions_train, interactions_val, interactions_test]:
    _df['u'] = _df['user_id'].map(user2idx)
    _df['i'] = _df['recipe_id'].map(item2idx)

# 4) Tạo PP_users đơn giản (aggregate từ interactions_train)
pp_users = (
    interactions_train.groupby('user_id')
    .agg(
        n_items=('recipe_id', 'nunique'),
        n_ratings=('rating', 'count'),
        avg_rating=('rating', 'mean')
    )
    .reset_index()
)
pp_users['u'] = pp_users['user_id'].map(user2idx)

# 5) Tạo PP_recipes đơn giản (copy từ recipes + map i)
pp_recipes = df_recipes.copy()
pp_recipes['i'] = pp_recipes['id'].map(item2idx)

# giữ recipes có trong mapping
pp_recipes = pp_recipes[pp_recipes['i'].notna()].copy()
pp_recipes['i'] = pp_recipes['i'].astype(int)

print("\n===== PP TABLES =====")
print("PP_users shape  :", pp_users.shape)
print("PP_recipes shape:", pp_recipes.shape)

# 6) Save outputs
# Nếu bạn muốn giữ đúng tên file hiện tại:
interactions_train.to_csv('../data/interactions_train.csv', index=False)
interactions_val.to_csv('../data/interactions_validation.csv', index=False)
interactions_test.to_csv('../data/interactions_test.csv', index=False)

pp_users.to_csv('../data/PP_users.csv', index=False)
pp_recipes.to_csv('../data/PP_recipes.csv', index=False)

print("\nSaved:")
print("- ../data/interactions_train.csv")
print("- ../data/interactions_validation.csv")
print("- ../data/interactions_test.csv")
print("- ../data/PP_users.csv")
print("- ../data/PP_recipes.csv")

===== JOIN CONSISTENCY =====
Interactions before join filter: 1,071,520
Interactions after join filter : 1,067,281
Removed (recipe_id not in df_recipes): 4,239

===== SPLIT SUMMARY =====
Train: 853,824
Val  : 106,728
Test : 106,729

===== PP TABLES =====
PP_users shape  : (112209, 5)
PP_recipes shape: (225530, 20)

Saved:
- ../data/interactions_train.csv
- ../data/interactions_validation.csv
- ../data/interactions_test.csv
- ../data/PP_users.csv
- ../data/PP_recipes.csv
